In [ ]:
# ! pip install selenium webdriver-manager beautifulsoup4 pandas

In [61]:
import time
import random
import re
import logging
import json
import pandas as pd

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementNotInteractableException,
    ElementClickInterceptedException,
)
from webdriver_manager.chrome import ChromeDriverManager


# Configuration
CITIES = [
    "New York, NY, USA",
    "Los Angeles, CA, USA",
    "San Francisco, CA, USA",
    "Chicago, IL, USA",
    "Washington, DC, USA",
]

LISTINGS_PER_CITY = 200
PAGE_PAUSE_MIN    = 1.5
PAGE_PAUSE_MAX    = 3

LISTING_PAUSE_MIN = 2
LISTING_PAUSE_MAX = 4

CITY_PAUSE_MIN    = 5
CITY_PAUSE_MAX    = 8
OUTPUT_FILE       = "scraped_airbnb_listings.csv"


# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)



# Driver
def build_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1440,900")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=options)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"},
    )
    return driver



# Helpers
def dismiss_popups(driver: webdriver.Chrome) -> None:
    """Safely dismiss cookie banners or modal pop-ups."""
    selectors = [
        '[data-testid="accept-btn"]',
        'button[aria-label="Close"]',
        'button[aria-label="close"]',
        '._148dgdpk',
        '[data-testid="modal-container"] button',
        'div[role="dialog"] button[aria-label="Close"]',
    ]
    for sel in selectors:
        try:
            for btn in driver.find_elements(By.CSS_SELECTOR, sel):
                try:
                    btn.click()
                except (ElementNotInteractableException, ElementClickInterceptedException):
                    driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.5)
        except Exception:
            pass


def slow_scroll(driver: webdriver.Chrome) -> None:
    """Gradually scroll to trigger lazy-loaded content."""
    height = driver.execute_script("return document.body.scrollHeight")
    pos = 0
    while pos < height:
        pos += 600
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(0.3)
        height = driver.execute_script("return document.body.scrollHeight")


def build_search_url(city: str, offset: int = 0) -> str:
    slug = city.replace(" ", "%20").replace(",", "%2C")
    return (
        f"https://www.airbnb.com/s/{slug}/homes"
        f"?tab_id=home_tab"
        f"&refinement_paths%5B%5D=%2Fhomes"
        f"&flexible_trip_lengths%5B%5D=one_week"
        f"&channel=EXPLORE"
        f"&pagination_search=true"
        f"&items_offset={offset}"
        f"&section_offset=3"
        f"&currency=USD"
    )



# Price parser

def parse_price_from_card(card_soup) -> int | None:
    try:
        # Airbnb price spans usually contain the currency symbol
        price_span = None

        for span in card_soup.find_all("span"):
            txt = span.get_text(strip=True)

            if re.search(r"[€$£¥]|EGP|AED", txt) and re.search(r"\d", txt):
                price_span = txt
                break

        if not price_span:
            return None

        text = price_span.replace("\xa0", " ")

        # extract numeric value
        m = re.search(r"([\d,]+(?:\.\d{1,2})?)", text)
        if not m:
            return None

        price = float(m.group(1).replace(",", ""))

        # check if total for multiple nights
        nights_match = re.search(r"for\s+(\d+)\s+night", text, re.I)

        if nights_match:
            nights = int(nights_match.group(1))
            if nights > 0:
                price = price / nights

        return int(round(price))

    except Exception:
        return None


# Collect listing URLs + price from search pages

def collect_urls(driver: webdriver.Chrome, city: str, target: int) -> list[dict]:
    """
    Paginate through Airbnb search results and collect listing URLs + prices.
    """

    results = []
    seen = set()
    page_num = 1

    next_url = build_search_url(city)

    log.info(f"[{city}] Collecting URLs (target={target}) ...")

    while len(results) < target:

        log.info(f"  Page {page_num} | collected={len(results)}")

        try:
            driver.get(next_url)

            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, 'div[data-testid="card-container"]')
                )
            )

            time.sleep(1)

        except TimeoutException:
            log.warning(f"  Timeout loading page {page_num}")
            break

        dismiss_popups(driver)
        slow_scroll(driver)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        cards = soup.find_all("div", attrs={"data-testid": "card-container"})
        log.info(f"  Found {len(cards)} cards")

        if not cards:
            log.warning("  No cards found — stopping for this city.")
            break

        new_count = 0

        for card in cards:

            anchor = card.find("a", href=re.compile(r"^/rooms/"))
            if not anchor:
                continue

            path = anchor["href"].split("?")[0]
            full_url = "https://www.airbnb.com" + path

            if full_url in seen:
                continue

            seen.add(full_url)

            price = parse_price_from_card(card)

            # property_type + neighbourhood from card 
            property_type = None
            neighbourhood = None

            title_tag = card.find(attrs={"data-testid": "listing-card-title"})
            if title_tag:
                title_text = title_tag.get_text(strip=True)

                if " in " in title_text.lower():
                    parts = re.split(r"\s+in\s+", title_text, maxsplit=1, flags=re.I)
                    property_type = parts[0].strip()
                    neighbourhood = parts[1].strip()
                else:
                    property_type = title_text.strip()

            results.append({
                "url": full_url,
                "price_per_night": price,
                "property_type": property_type,
                "neighbourhood": neighbourhood
            })

            new_count += 1

            if len(results) >= target:
                break

        log.info(f"  Added {new_count} listings (total={len(results)})")

        if new_count == 0:
            log.info("  No new listings — stopping pagination.")
            break

        # Find the next page link
        next_link = None

        # Try common aria-label variants
        for label in ["Next", "Next page", "next"]:
            next_link = soup.find("a", attrs={"aria-label": label})
            if next_link:
                break

        # Fallback: visible link text
        if not next_link:
            next_link = soup.find("a", string=re.compile(r"next", re.I))

        # Fallback: detect offset pagination
        if not next_link:
            next_link = soup.find("a", href=re.compile(r"items_offset=\d+"))

        if next_link and next_link.get("href"):

            next_url = "https://www.airbnb.com" + next_link["href"]

            log.info(f"  Moving to next page → {next_url}")

        else:

            log.info("  No next page found — stopping.")
            break

        page_num += 1

        time.sleep(random.uniform(PAGE_PAUSE_MIN, PAGE_PAUSE_MAX))

    log.info(f"[{city}] Collected {len(results)} URLs.\n")

    return results


# Amenities helpers

def click_show_all_amenities(driver: webdriver.Chrome) -> bool:
    """Click the 'Show all X amenities' button. Returns True if modal opened."""
    try:
        # XPath targets button containing a span with data-button-content="true"
        # and text that includes "amenities"
        btns = driver.find_elements(
            By.XPATH,
            '//button[.//span[@data-button-content="true" '
            'and contains(., "amenities")]]'
        )
        if not btns:
            # Broader fallback
            btns = driver.find_elements(
                By.XPATH, '//button[contains(., "amenities")]'
            )
        if btns:
            driver.execute_script("arguments[0].scrollIntoView(true);", btns[0])
            time.sleep(0.5)
            try:
                btns[0].click()
            except (ElementNotInteractableException, ElementClickInterceptedException):
                driver.execute_script("arguments[0].click();", btns[0])
            time.sleep(2)
            return True
    except Exception:
        pass
    return False


def extract_amenities_from_modal(driver: webdriver.Chrome) -> list[str]:
    """
    Parse amenity names from the open modal.
    """
    amenities = []
    try:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        # Find all divs whose id ends with "-row-title", that's the pattern for all amenity name divs
        title_divs = soup.find_all(
            "div",
            id=re.compile(r"-row-title$"),
            class_=lambda c: c and "twad414" in c
        )
        for div in title_divs:
            text = div.get_text(strip=True)
            if text and len(text) > 1:
                amenities.append(text)
    except Exception:
        pass
    return amenities


def close_amenities_modal(driver: webdriver.Chrome) -> None:
    """Close the amenities modal."""
    try:
        # The modal close button is typically aria-label="Close" inside role="dialog"
        btns = driver.find_elements(
            By.XPATH,
            '//div[@role="dialog"]//button[@aria-label="Close" '
            'or @aria-label="close"]'
        )
        if btns:
            btns[0].click()
            time.sleep(1)
        else:
            # Press Escape as fallback
            driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
            time.sleep(1)
    except Exception:
        pass


def parse_ol_summary(soup: BeautifulSoup) -> dict:
    
    result = {
        "guests": None, "bedrooms": None,
        "beds": None,   "bathrooms": None,
        "bathroom_type": None,
    }
    try:
        ol = soup.find("ol", class_=lambda c: c and "lgx66tx" in c)
        if not ol:
            return result

        for li in ol.find_all("li", class_=lambda c: c and "l7n4lsf" in c):
            # Strip the aria-hidden separator spans (· dots) and get clean text
            for hidden in li.find_all("span", attrs={"aria-hidden": "true"}):
                hidden.decompose()
            text = li.get_text(" ", strip=True).lower()

            num_match = re.search(r"(\d+)", text)
            num = int(num_match.group(1)) if num_match else None

            if "guest" in text and num:
                result["guests"] = num
            elif "bedroom" in text and num:
                result["bedrooms"] = num
            elif "studio" in text:
                result["bedrooms"] = 0
            elif "bed" in text and "bedroom" not in text and num:
                result["beds"] = num
            elif "shared" in text and "bath" in text:
                result["bathrooms"]     = 0.5
                result["bathroom_type"] = "shared"
            elif "bath" in text:
                result["bathrooms"]     = num if num else 1
                result["bathroom_type"] = "private"
    except Exception:
        pass
    return result

# Extract full details from individual listing page

def extract_listing(driver, url, city, price_from_card,
                    property_type_from_card, neighbourhood_from_card):
    """
    Visit a single listing page and extract all fields.
    price_from_card is already known from Step 1 — no need to re-parse price.
    """
    data = {
        "city":              city,
        "listing_url":       url,
        "neighbourhood":     neighbourhood_from_card,
        "property_type":     property_type_from_card,
        "room_type":         None,
        "guests":            None,
        "bedrooms":          None,
        "beds":              None,
        "bathrooms":         None,    # 0.5 = shared
        "bathroom_type":     None,    # "shared" | "private"
        "price_per_night":   price_from_card,   # already parsed from card
        "rating":            None,
        "review_count":      None,
        "is_superhost":      False,
        "free_cancellation": False,
        "amenities":         None,
        "latitude":          None,
        "longitude":         None,
        "host_response_rate":None,
        "host_identity_verified": False
    }

    try:
        driver.get(url)

        # Wait for the summary <ol class="lgx66tx"> — confirmed in real HTML
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "ol.lgx66tx")
                )
            )
        except TimeoutException:
            # Fallback wait: just wait for h1
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )

        time.sleep(1.5)
        dismiss_popups(driver)
        slow_scroll(driver)
        time.sleep(1.5)

        soup      = BeautifulSoup(driver.page_source, "html.parser")
        page_text = soup.get_text(" ", strip=True)

        # guests / bedrooms / beds / bathrooms 
        summary = parse_ol_summary(soup)
        data.update({
            "guests":        summary["guests"],
            "bedrooms":      summary["bedrooms"],
            "beds":          summary["beds"],
            "bathrooms":     summary["bathrooms"],
            "bathroom_type": summary["bathroom_type"],
        })

        # room type
        try:
            ROOM_TYPES = re.compile(
                r"^(Entire|Private room|Shared room|Hotel room|Room in)",
                re.I
            )
            for tag in soup.find_all(["h2", "span", "div"]):
                txt = tag.get_text(strip=True)
                if ROOM_TYPES.match(txt) and len(txt) < 80:
                    # Extract only the room type (before " in ")
                    m = re.match(r"^(.*?)\s+in\s+", txt)

                    if m:
                        data["room_type"] = m.group(1).strip()
                    else:
                        # fallback if "in" pattern doesn't exist
                        data["room_type"] = txt.strip()
                    break
        except Exception:
            pass

        # latitude / longitude (from ld+json schema) 
        try:
            scripts = soup.find_all("script", type="application/ld+json")

            for s in scripts:
                try:
                    js = json.loads(s.string)

                    if isinstance(js, dict) and js.get("@type") == "VacationRental":
                        data["latitude"] = js.get("latitude")
                        data["longitude"] = js.get("longitude")

                        # sometimes rating is here too (more reliable)
                        if "aggregateRating" in js:
                            rating = js["aggregateRating"].get("ratingValue")
                            count = js["aggregateRating"].get("ratingCount")

                            if rating:
                                data["rating"] = float(rating)

                            if count:
                                data["review_count"] = int(count)

                        break
                except:
                    continue
        except:
            pass


        # host response rate 
        try:
            m = re.search(r"Response rate:\s*(\d+)%", page_text, re.I)

            if m:
                data["host_response_rate"] = int(m.group(1))
        except:
            pass

        # host identity verified 
        try:
            verified_tag = soup.find(
                attrs={"aria-label": re.compile(r"identity verified", re.I)}
            )

            if verified_tag:
                data["host_identity_verified"] = True
        except:
            pass

        # rating 
        try:
            for tag in soup.find_all(["span", "div"]):
                txt = tag.get_text(strip=True)
                if re.fullmatch(r"[1-5]\.\d{1,2}", txt):
                    data["rating"] = float(txt)
                    break
        except Exception:
            pass

        # review count 
        try:
            m = re.search(r"(\d{1,5})\s+review", page_text, re.I)
            if not m:
                m = re.search(r"\((\d{1,5})\)", page_text)
            if m:
                data["review_count"] = int(m.group(1))
        except Exception:
            pass

        # superhost
        try:
            data["is_superhost"] = bool(re.search(r"superhost", page_text, re.I))
        except Exception:
            pass

        # free cancellation 
        try:
            fc = soup.find("div", class_=lambda c: c and "t789gv3" in c)
            if fc:
                data["free_cancellation"] = "free cancellation" in fc.get_text(strip=True).lower()
            else:
                # Fallback: plain text search
                data["free_cancellation"] = bool(
                    re.search(r"free cancellation", page_text, re.I)
                )
        except Exception:
            pass

        # amenities — open modal and scrape all items 
        try:
            opened = click_show_all_amenities(driver)
            if opened:
                amenity_list = extract_amenities_from_modal(driver)
                if amenity_list:
                    data["amenities"] = ", ".join(amenity_list)
                close_amenities_modal(driver)
        except Exception:
            pass

    except TimeoutException:
        log.warning(f"  Timeout loading: {url}")
    except Exception as e:
        log.warning(f"  Error scraping {url}: {e}")

    return data


# Main

def main():
    all_data = []
    driver   = build_driver()

    try:
        for city in CITIES:
            log.info(f"\n{'='*55}")
            log.info(f"  CITY: {city}")
            log.info(f"{'='*55}")

            # Collect URLs + prices from search cards
            url_records = collect_urls(driver, city, target=LISTINGS_PER_CITY)

            # Visit each listing for remaining fields
            log.info(f"[{city}] Scraping {len(url_records)} listing pages ...")
            for i, rec in enumerate(url_records, start=1):
                log.info(f"  [{i}/{len(url_records)}] {rec['url']}")
                
                record = extract_listing(
                            driver,
                            rec["url"],
                            city,
                            rec["price_per_night"],
                            rec["property_type"],
                            rec["neighbourhood"],
                        )
                all_data.append(record)

                # Incremental save every 50 listings
                if i % 50 == 0:
                    pd.DataFrame(all_data).to_csv(
                        OUTPUT_FILE, index=False, encoding="utf-8-sig"
                    )
                    log.info(f"  💾 Auto-saved {len(all_data)} records.")

                time.sleep(random.uniform(LISTING_PAUSE_MIN, LISTING_PAUSE_MAX))

            log.info(f"[{city}] Done. Pausing before next city ...")
            time.sleep(random.uniform(CITY_PAUSE_MIN, CITY_PAUSE_MAX))

    finally:
        driver.quit()
        log.info("Browser closed.")

    # Final save
    df = pd.DataFrame(all_data, columns=[
    "city",
    "property_type",
    "neighbourhood",
    "listing_url",
    "room_type",
    "guests",
    "bedrooms",
    "beds",
    "bathrooms",
    "bathroom_type",
    "price_per_night",
    "rating",
    "review_count",
    "latitude",
    "longitude",
    "is_superhost",
    "host_response_rate",
    "host_identity_verified",
    "free_cancellation",
    "amenities",
])
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    log.info(f"\nDone! {len(df)} listings → '{OUTPUT_FILE}'")
    log.info(f"\nCity breakdown:\n{df['city'].value_counts().to_string()}")
    log.info(f"\nMissing values:\n{df.isnull().sum().to_string()}")
    log.info(f"\nSample:\n{df.head(3).to_string()}")


if __name__ == "__main__":
    main()

16:19:30  INFO  ====== WebDriver manager ======
16:19:30  INFO  Get LATEST chromedriver version for google-chrome
16:19:31  INFO  Get LATEST chromedriver version for google-chrome
16:19:31  INFO  Driver [/Users/mac/.wdm/drivers/chromedriver/mac64/146.0.7680.165/chromedriver-mac-x64/chromedriver] found in cache
16:19:33  INFO  
16:19:33  INFO    CITY: Washington, DC, USA
16:19:33  INFO  =======================================================
16:19:33  INFO  [Washington, DC, USA] Collecting URLs (target=200) ...
16:19:33  INFO    Page 1 | collected=0
16:19:44  INFO    Found 18 cards
16:19:44  INFO    Added 18 listings (total=18)
16:19:44  INFO    Moving to next page → https://www.airbnb.com/s/Washington%252C-DC%252C-USA/homes?refinement_paths%5B%5D=%2Fhomes&flexible_trip_lengths%5B%5D=one_week&channel=EXPLORE&pagination_search=true&query=Washington%252C%20DC%252C%20USA&place_id=ChIJW-T2Wt7Gt4kRKl2I1CJFUsI&monthly_start_date=2026-05-01&monthly_length=3&monthly_end_date=2026-08-01&price_fi

In [67]:
df= pd.read_csv(OUTPUT_FILE)
df.head()

,city,property_type,neighbourhood,listing_url,room_type,guests,bedrooms,beds,bathrooms,bathroom_type,price_per_night,rating,review_count,latitude,longitude,is_superhost,host_response_rate,host_identity_verified,free_cancellation,amenities
0,"Los Angeles, CA, USA",Home,Malibu,https://www.airbnb.com/rooms/1596739057740115269,Entire home,8.0,2.0,5.0,2.0,private,1258,5.00,39.0,34.06684,-118.67309,True,100.0,True,False,"Bathtub, Hair dryer, Cleaning products, Shampo..."
1,"Los Angeles, CA, USA",Apartment,Vernon,https://www.airbnb.com/rooms/1640195326455382137,Entire rental unit,3.0,1.0,2.0,1.0,private,291,4.77,2.0,34.01345,-118.23624,True,100.0,True,False,"Valley view, City skyline view, Bathtub, Hair ..."
2,"Los Angeles, CA, USA",Home,Los Angeles,https://www.airbnb.com/rooms/1220699370253252798,Entire home,10.0,4.0,8.0,3.0,private,500,5.00,1.0,34.18719,-118.37326,False,100.0,True,False,"Garden view, Hair dryer, Cleaning products, Sh..."
3,"Los Angeles, CA, USA",Tiny home,Topanga,https://www.airbnb.com/rooms/984107417862937992,NaN,2.0,1.0,1.0,1.0,private,314,4.96,290.0,34.07873,-118.60588,True,100.0,True,False,"Shampoo, Conditioner, Body soap, Outdoor showe..."
4,"Los Angeles, CA, USA",Home,Los Angeles County,https://www.airbnb.com/rooms/1590928612173170115,Entire home,4.0,3.0,3.0,3.0,private,377,5.00,7.0,34.09098,-118.60125,False,100.0,True,False,"Bathtub, Hair dryer, Cleaning products, Shampo..."
